In [ ]:
import os
import numpy as np
import nibabel as nib
from scipy.interpolate import interp1d
import torch
from nibabel.processing import resample_from_to

In [ ]:
shape_target = (96, 96, 96)
affine_target = np.array([
    [-2,  0,  0,   96],
    [ 0,  2,  0, -112],
    [ 0,  0,  2,  -90],
    [ 0,  0,  0,    1]
])

MNI152_brainmask = './MNI152_T1_1mm_brain_mask.nii.gz'

In [ ]:
def resample2pt(img, savedir, t_r = None, brain_mask = None):

    img = nib.load(img)
    os.makedirs(savedir, exist_ok = True)
    if t_r == None: t_r = img.header['pixdim'][4]
    if brain_mask == None: brain_mask = MNI152_brainmask
    spatial_resample = False if (abs(img.affine.diagonal())[:3] == abs(affine_target.diagonal())[:3]).all() else True
    temporal_resample = False if t_r  >= 0.7 and t_r <= 0.8 else True

    if spatial_resample == True:
        data = resample_from_to(img, ((*shape_target, img.header['dim'][4]), affine_target), order = 3).get_fdata(dtype = np.float32)
    else:
        data = img.get_fdata(dtype = np.float32); affine = img.affine
        if affine[0, 0] == -affine_target[0, 0]: data = np.flip(data, axis = 0); affine[0, 3] = affine[0, 3] + data.shape[0] * affine[0, 0]; affine[0, 0] = affine_target[0, 0]
        if affine[1, 1] == -affine_target[1, 1]: data = np.flip(data, axis = 1); affine[1, 3] = affine[1, 3] + data.shape[1] * affine[1, 1]; affine[1, 1] = affine_target[1, 1]
        if affine[2, 2] == -affine_target[2, 2]: data = np.flip(data, axis = 2); affine[2, 3] = affine[2, 3] + data.shape[2] * affine[2, 2]; affine[2, 2] = affine_target[2, 2]
        dx1 = round((affine[0, 3] - affine_target[0, 3]) / affine[0, 0]); dx2 = round(shape_target[0] - data.shape[0] - dx1)
        dy1 = round((affine[1, 3] - affine_target[1, 3]) / affine[1, 1]); dy2 = round(shape_target[1] - data.shape[1] - dy1)
        dz1 = round((affine[2, 3] - affine_target[2, 3]) / affine[2, 2]); dz2 = round(shape_target[2] - data.shape[2] - dz1)
        data = np.pad(data, [(np.maximum(dx1, 0), np.maximum(dx2, 0)), (np.maximum(dy1, 0), np.maximum(dy2, 0)), (np.maximum(dz1, 0), np.maximum(dz2, 0)), (0, 0)], mode = 'constant', constant_values = 0)
        data = data[abs(dx1) if dx1 < 0 else None : dx2 if dx2 < 0 else None, abs(dy1) if dy1 < 0 else None : dy2 if dy2 < 0 else None, abs(dz1) if dz1 < 0 else None : dz2 if dz2 < 0 else None]
    
    if temporal_resample == True:
        shape_space = data.shape[:-1]
        t_orig = np.arange(data.shape[-1]) * t_r
        t_new = np.arange(0, t_orig[-1] + 0.72, 0.72); t_new = t_new[t_new <= t_orig[-1]]
        data_flat = data.reshape(-1, data.shape[-1])
        new_data_flat = np.empty((data_flat.shape[0], len(t_new)), dtype = np.float32)
        for idx in range(data_flat.shape[0]): f = interp1d(t_orig, data_flat[idx], kind = 'cubic', fill_value = 'extrapolate'); new_data_flat[idx] = f(t_new)
        data = new_data_flat.reshape(*shape_space, len(t_new))
    
    mask = resample_from_to(nib.load(brain_mask), ((96, 96, 96), affine_target), order = 0).get_fdata(dtype = np.float32)
    data_avg = np.mean(data[mask == 1]); data_std = np.std(data[mask == 1])
    data[mask == 1] = (data[mask == 1] - data_avg) / data_std; data[mask == 0] = 0

    for i in range(data.shape[-1]):
        temp = data[..., i].copy(); tensor = torch.from_numpy(temp).to(torch.float32)
        torch.save(tensor, '{}/{:04d}.pt'.format(savedir, i))

    return data_avg, data_std